# Scene visualizer

Renders **every** grounding case in one or more scenes — correct and wrong alike. IoU is
not a filter here, only a colour.

| box | colour |
|---|---|
| ground truth | green |
| prediction, IoU >= 0.25 | purple |
| prediction, IoU < 0.25 | red |

Each case becomes one PNG with the three-panel layout used by the paper figures: room top
view, 3-D view from the room corner, and the verdict. Output goes to
`vis_results/<scene_id>/`, one folder per scene.


In [ ]:
# One name, several names, or "all". Leave empty to be prompted.
#   SCENE_IDS = ["scene0025_00"]
#   SCENE_IDS = ["scene0011_00", "scene0025_00", "scene0019_00"]
#   SCENE_IDS = ["all"]        # 141 scenes, ~2.7 h and ~5.3 GB at dpi 110
SCENE_IDS      = ['all']

PREDICTIONS    = "outputs/2024-12-18_20-40-38_3DVG-FIXED/predictions.p"
IOU_THRESHOLD  = 0.25        # at or above -> purple, below -> red
DPI            = 200         # 110 is quicker; 200 for print (~1.7x slower, ~3x the disk)
OUT_ROOT       = "vis_results"
SKIP_EXISTING  = True        # True: leave scenes that already have an index.json alone

# Where the points come from. "ply" is the original scan (44k-255k points, mean ~160k);
# "npy" is the detector's subsampled cloud (exactly 50k, and no more to be had).
SOURCE         = "ply"
SCANS_ROOT     = "data/scannet/scans"
MAX_POINTS     = 500000      # draws every point of nearly every scan; lower it to speed up

In [ ]:
import os, sys, json, difflib, pickle

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

if not os.path.isfile("experiments/analysis.py"):        # be runnable from a subdirectory
    os.chdir(os.path.dirname(os.path.abspath("")) or ".")
sys.path.insert(0, os.getcwd())

from experiments.analysis import (
    load_scene, load_points as _load_points, same_class_boxes, verify_frames,
    box_center, draw_box_2d, look_at, project, draw_box_projected,
    GT_COLOR, DISTRACTOR_COLOR, ensure_dir)


def load_points(scene_id):
    """This notebook's SOURCE / SCANS_ROOT applied to the shared loader."""
    return _load_points(scene_id, SOURCE, SCANS_ROOT)

PRED_OK_COLOR   = "#7B4BC9"      # purple: IoU >= threshold
PRED_BAD_COLOR  = "#D62828"      # red:    IoU <  threshold
print("helpers imported from experiments/analysis.py")
print(f"point source: {SOURCE!r}"
      + (f" ({SCANS_ROOT})" if SOURCE == "ply" else " (data/scannet/scannet_data)"))

In [ ]:
with open(PREDICTIONS, "rb") as handle:
    PREDICTIONS_DATA = pickle.load(handle)

def cases_in(scene_id):
    """Every (object_id, ann_id, iou, gt, pred) in one scene, worst IoU first."""
    found = []
    for object_id, annotations in (PREDICTIONS_DATA.get(scene_id) or {}).items():
        for ann_id, entry in (annotations or {}).items():
            if isinstance(entry, dict) and "iou" in entry:
                found.append({"object_id": str(object_id), "ann_id": str(ann_id),
                              "iou": float(entry["iou"]),
                              "gt": np.asarray(entry["gt_bbox"], dtype=float),
                              "pred": np.asarray(entry["pred_bbox"], dtype=float)})
    return sorted(found, key=lambda case: case["iou"])

SCENES = sorted(scene for scene in PREDICTIONS_DATA
                if isinstance(PREDICTIONS_DATA[scene], dict) and cases_in(scene))
print(f"{len(SCENES)} scenes in {PREDICTIONS}")
print("examples:", ", ".join(SCENES[:4]), "...")

In [ ]:
def resolve_scenes(requested):
    """SCENE_IDS when set, else prompt. Returns the list of usable scene ids."""
    if not requested:
        try:
            typed = input("scene id(s), comma-separated, or 'all': ").strip()
        except (EOFError, RuntimeError):
            print("No stdin here -- set SCENE_IDS in the config cell instead.")
            return []
        requested = [part.strip() for part in typed.split(",") if part.strip()]

    if any(name.lower() == "all" for name in requested):
        print(f"all {len(SCENES)} scenes selected")
        return list(SCENES)

    wanted, seen = [], set()
    for name in requested:
        if name in seen:
            continue
        seen.add(name)
        if name in SCENES:
            wanted.append(name)
            continue
        print(f"{name!r} is not in the predictions ({len(SCENES)} scenes available).")
        near = difflib.get_close_matches(name, SCENES, n=5, cutoff=0.4)
        if near:
            print("   did you mean:", ", ".join(near))
    return wanted


TARGETS = resolve_scenes(SCENE_IDS)
if TARGETS:
    total = sum(len(cases_in(scene)) for scene in TARGETS)
    print(f"\n{len(TARGETS)} scene(s), {total} case(s) total "
          f"-- roughly {total * 0.79 * (DPI / 110) / 60:.1f} min at {DPI} dpi")
else:
    print("nothing to render")

In [ ]:
def render_one(scene_id, case, xyz, rgb, out_dir, dpi=DPI, max_points=MAX_POINTS):
    """One case: room top view, 3-D corner view, verdict. Returns the path written."""
    gt, pred, iou = case["gt"], case["pred"], case["iou"]
    correct = iou >= IOU_THRESHOLD
    pred_color = PRED_OK_COLOR if correct else PRED_BAD_COLOR

    figure = plt.figure(figsize=(14.5, 5.2))
    grid = figure.add_gridspec(1, 3, width_ratios=[1.0, 1.45, 0.85], wspace=0.13)

    ax1 = figure.add_subplot(grid[0, 0])
    step = max(1, len(xyz) // max(max_points, 1))
    sampled, sampled_rgb = xyz[::step], (rgb[::step] if rgb is not None else None)
    if sampled_rgb is not None:
        sampled_rgb = np.clip(sampled_rgb, 0, 1) ** 0.65
    ax1.scatter(sampled[:, 0], sampled[:, 1], s=2.0,
                c=sampled_rgb if sampled_rgb is not None else "#B8BEC9",
                linewidths=0, rasterized=True)
    distractors = same_class_boxes(scene_id, case["object_id"])
    for box in distractors:
        draw_box_2d(ax1, box, DISTRACTOR_COLOR, linewidth=0.9, alpha=0.75)
    draw_box_2d(ax1, gt, GT_COLOR, linewidth=2.0, fill=True)
    draw_box_2d(ax1, pred, pred_color, linewidth=2.0, fill=True)
    drift = float(np.linalg.norm(box_center(gt) - box_center(pred)))
    ax1.annotate("", xy=box_center(pred)[:2], xytext=box_center(gt)[:2],
                 arrowprops=dict(arrowstyle="->", color="#333333",
                                 linewidth=1.2, linestyle=":"))
    ax1.set_title(f"{scene_id} -- top view   (centre error {drift:.2f} m)", fontsize=9)
    ax1.set_aspect("equal")
    ax1.set_xlabel("x (m)", fontsize=8)
    ax1.set_ylabel("y (m)", fontsize=8)
    ax1.tick_params(labelsize=7)

    scene_low, scene_high = xyz.min(axis=0), xyz.max(axis=0)
    diagonal = float(np.linalg.norm(scene_high - scene_low))
    eye = np.array([scene_high[0] + 0.30 * diagonal,
                    scene_low[1] - 0.30 * diagonal,
                    scene_high[2] + 0.30 * diagonal])
    rotation = look_at(eye, (scene_low + scene_high) / 2.0)

    shown, shown_rgb = xyz[::step], (rgb[::step] if rgb is not None else None)
    if shown_rgb is not None:
        shown_rgb = np.clip(shown_rgb, 0, 1) ** 0.6

    ax2 = figure.add_subplot(grid[0, 1])
    u, v, depth = project(shown, eye, rotation)
    visible = depth > 0
    order = np.argsort(-depth[visible])               # far to near: painter's algorithm
    u, v = u[visible][order], v[visible][order]
    ax2.scatter(u, v, s=2.2,
                c=shown_rgb[visible][order] if shown_rgb is not None else "#B8BEC9",
                linewidths=0, rasterized=True)
    draw_box_projected(ax2, gt, eye, rotation, GT_COLOR)
    draw_box_projected(ax2, pred, eye, rotation, pred_color)
    if len(u):
        pad = 0.04 * max(u.max() - u.min(), 1.0)
        ax2.set_xlim(u.min() - pad, u.max() + pad)
        ax2.set_ylim(v.max() + pad, v.min() - pad)    # screen coordinates run downward
    ax2.set_title(f"3-D view from the room corner  ({len(u):,} points)", fontsize=9)
    ax2.set_aspect("equal")
    ax2.axis("off")

    ax3 = figure.add_subplot(grid[0, 2])
    ax3.axis("off")
    verdict = "CORRECT" if correct else "WRONG"
    lines = [("verdict", f"{verdict}  (IoU {'>=' if correct else '<'} {IOU_THRESHOLD})"),
             ("IoU", f"{iou:.3f}"),
             ("centre error", f"{drift:.2f} m"),
             ("same-label boxes drawn", str(len(distractors)))]
    ax3.text(0.0, 1.0, "\n".join(f"{label:<22}: {value}" for label, value in lines),
             va="top", ha="left", fontsize=8.4, family="monospace",
             transform=ax3.transAxes)
    ax3.text(0.0, 0.02,
             f"green = ground truth\n{'purple' if correct else 'red'} = prediction"
             f"\ngrey = other objects of the same class",
             fontsize=7.6, color="#444444", transform=ax3.transAxes)

    offset = verify_frames(scene_id, case["object_id"], gt)
    if offset is not None and offset > 0.05:
        figure.text(0.5, 0.015,
                    f"WARNING: gt box centre is {offset:.3f} m from the scene's "
                    f"aligned_bbox entry", ha="center", color=PRED_BAD_COLOR, fontsize=9)

    figure.suptitle(f"{scene_id}  object {case['object_id']}  annotation {case['ann_id']}"
                    f"  --  {verdict}  --  IoU {iou:.2f}", fontsize=10.5)
    figure.subplots_adjust(left=0.045, right=0.99, top=0.88, bottom=0.12)

    name = f"{iou:.3f}_obj{case['object_id']}_ann{case['ann_id']}.png"
    path = os.path.join(out_dir, name)
    figure.savefig(path, dpi=dpi)
    plt.close(figure)
    return path, name

In [ ]:
def render_scene(scene_id):
    """Render every case in one scene. Returns (num_written, num_wrong) or None if skipped."""
    out_dir = os.path.join(OUT_ROOT, scene_id)
    if SKIP_EXISTING and os.path.isfile(os.path.join(out_dir, "index.json")):
        print(f"{scene_id}: already rendered, skipped")
        return None

    xyz, rgb, source = load_points(scene_id)
    if xyz is None:
        print(f"{scene_id}: no point cloud found (tried {SOURCE}) -- skipped")
        return None

    cases = cases_in(scene_id)
    ensure_dir(out_dir)
    written = []
    for case in cases:
        path, name = render_one(scene_id, case, xyz, rgb, out_dir)
        written.append({"file": name, "object_id": case["object_id"],
                        "ann_id": case["ann_id"], "iou": case["iou"],
                        "verdict": "CORRECT" if case["iou"] >= IOU_THRESHOLD else "WRONG"})

    num_wrong = sum(1 for item in written if item["verdict"] == "WRONG")
    with open(os.path.join(out_dir, "index.json"), "w") as handle:
        json.dump({"scene_id": scene_id, "predictions": PREDICTIONS,
                   "iou_threshold": IOU_THRESHOLD, "dpi": DPI,
                   "source": source, "num_points": int(len(xyz)),
                   "num_cases": len(written),
                   "num_correct": len(written) - num_wrong,
                   "num_wrong": num_wrong, "cases": written}, handle, indent=2)
    return len(written), num_wrong, len(xyz)


summary = []
for position, scene_id in enumerate(TARGETS, 1):
    print(f"[{position}/{len(TARGETS)}] {scene_id}", flush=True)
    result = render_scene(scene_id)
    if result is None:
        continue
    count, num_wrong, num_points = result
    summary.append({"scene_id": scene_id, "cases": count,
                    "correct": count - num_wrong, "wrong": num_wrong,
                    "num_points": num_points})
    print(f"    {count} case(s) -> {OUT_ROOT}/{scene_id}  "
          f"({count - num_wrong} correct, {num_wrong} wrong, "
          f"{num_points:,} points)", flush=True)

if summary:
    cases = sum(item["cases"] for item in summary)
    correct = sum(item["correct"] for item in summary)
    print(f"\n{len(summary)} scene(s), {cases} figure(s): "
          f"{correct} correct, {cases - correct} wrong "
          f"({100 * correct / cases:.1f}% at IoU >= {IOU_THRESHOLD})")
    with open(os.path.join(ensure_dir(OUT_ROOT), "summary.json"), "w") as handle:
        json.dump({"predictions": PREDICTIONS, "iou_threshold": IOU_THRESHOLD,
                   "dpi": DPI, "source": SOURCE, "scenes": summary}, handle, indent=2)
    print(f"wrote {OUT_ROOT}/summary.json")